# Run the NRQC quantum calculator

Why do four readings appear when we look for the repeating powers of 2 modulo 15? Which vowel pairs produce the same Sanskrit sound? Run the cells in order to explore both questions in a CPU notebook. The last cell decodes a two-bit parity check. All three are ideal software calculations.

[Software release v1.2.0](https://github.com/Parveen117/quantum-classical-provisional/releases/tag/v1.2.0) · [Archived source and citation](https://doi.org/10.5281/zenodo.24624377)

## Load the released calculator

On a fresh Colab runtime this cell downloads the wheel attached to the public release and checks its SHA-256 before installation. For an offline run, place that wheel in the notebook's current directory and install NumPy 1.26 or later in the 1.x or 2.x series beforehand. Use Python 3.11 or later.

In [1]:
import hashlib
import subprocess
import sys
import tempfile
from importlib.metadata import version
from pathlib import Path
from urllib.request import urlopen

filename = "native_relational_qc-1.2.0-py3-none-any.whl"
url = "https://github.com/Parveen117/quantum-classical-provisional/releases/download/v1.2.0/" + filename
expected = "b7f6c6b0a4020d953b47f32eae3ea5255d87b27b904f20552962400571dd78b1"
local = Path.cwd() / filename
if local.is_file():
    payload = local.read_bytes()
else:
    with urlopen(url, timeout=60) as response:
        payload = response.read()
if hashlib.sha256(payload).hexdigest() != expected:
    raise ValueError("Wheel checksum differs from the v1.2.0 release. Check the file before installing.")
with tempfile.TemporaryDirectory() as temp_dir:
    verified = Path(temp_dir) / filename
    verified.write_bytes(payload)
    subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--quiet", str(verified)], check=True)
assert version("native-relational-qc") == "1.2.0"
print("NRQC v1.2.0 ready. Release SHA-256 verified.")

NRQC v1.2.0 ready. Release SHA-256 verified.


## 1. Read a repeating pattern

Starting at 1, repeatedly multiply by 2 and keep the remainder after division by 15. The values go 1, 2, 4, 8, then back to 1. A three-bit phase reading has eight possible labels. A period of four predicts peaks at labels 0, 2, 4 and 6, spaced evenly across the eight labels. This cell reuses one control wire three times and checks every resulting probability against the Fourier formula. The check uses exact arithmetic. The printed probabilities are decimals.

In [2]:
from decimal import Decimal
from native_relational_qc.record_phase import build_record, calculate_record

phase = calculate_record(build_record(modulus=15, base=2, precision=3))
assert phase["status"] == "completed" and phase["assessment"]["exact_fourier_law_agrees"]
for bits, entry in phase["record"]["probabilities"].items():
    print(f"{bits}  label {int(bits, 2)}  probability {Decimal(entry['decimal']):.2f}")

000  label 0  probability 0.25
001  label 1  probability 0.00
010  label 2  probability 0.25
011  label 3  probability 0.00
100  label 4  probability 0.25
101  label 5  probability 0.00
110  label 6  probability 0.25
111  label 7  probability 0.00


## 2. Work backwards from a Sanskrit sound

The example uses two Pāṇinian sandhi rules, 6.1.101 and 6.1.77, on the four vowels इ, ई, उ and ऊ. There are 16 ordered input pairs. The forward calculation joins two vowels. The reverse calculation uses one Grover iteration to give extra probability to pairs that produce the chosen sound. Change `surface` from `"ई"` to `"यु"` to see how having fewer matching pairs changes the outcome.

In [3]:
from native_relational_qc.panini_sandhi import join, search

_, joined = join("ई", "इ")
print(f"ī + i → {joined['surface']}  (rule {joined['rule']})")
surface = "ई"
_, found = search(surface)
for pair in found["candidates"]:
    if pair["matches"]:
        print(f"{pair['left']} + {pair['right']}  probability {pair['probability']}")
print("Total probability on matching pairs:", found["uniform_matching_probability"], "→", found["matching_probability"])

ī + i → ī  (rule 6.1.101)
i + i  probability 1/4
i + ī  probability 1/4
ī + i  probability 1/4
ī + ī  probability 1/4
Total probability on matching pairs: 1/4 → 1


## 3. Decode two hidden bits

The first check tells whether the hidden bits differ. The second check gives the right-hand bit. A check of `01` means that the bits agree and the right-hand bit is 1, so the hidden pair must be `11`. Compare that prediction with the four ideal circuit outputs below.

In [4]:
from native_relational_qc.grover_cloud import ideal

for syndrome, outcomes in ideal(parity=True)["probabilities"].items():
    winner = max(outcomes, key=outcomes.get)
    print(f"check {syndrome} → hidden pair {winner}  (ideal probability {outcomes[winner]:.3f})")

check 00 → hidden pair 00  (ideal probability 1.000)
check 01 → hidden pair 11  (ideal probability 1.000)
check 10 → hidden pair 10  (ideal probability 1.000)
check 11 → hidden pair 01  (ideal probability 1.000)


### Explore the twelve-bit audit

For three larger inputs, the archived evidence compares all 4,096 output bins from a recycled control with a full-register circuit. The linked JSON gives a row per input. The reproduction script accepts the original compressed full-register outputs. The small examples above finish independently of that longer audit.

[Audit data](https://github.com/Parveen117/quantum-classical-provisional/blob/67478206858dac407198bcd58cc3c3c5663dfd6a/native_relational_qc/record_full_12_audit.json) · [Reproduction script](https://github.com/Parveen117/quantum-classical-provisional/blob/67478206858dac407198bcd58cc3c3c5663dfd6a/benchmarks/record_full_12_audit.py)